# 04 Business Momentum Weekly Trends
Classify business momentum with rolling windows and threshold experiments.

In [1]:
import warnings
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore")
PROJECT_ROOT = Path.cwd().resolve()

if not (PROJECT_ROOT / "utils" / "utils.py").exists():

    if (PROJECT_ROOT / "notebooks" / "utils" / "utils.py").exists():
        PROJECT_ROOT = PROJECT_ROOT / "notebooks"

    elif (PROJECT_ROOT.parent / "utils" / "utils.py").exists():
        PROJECT_ROOT = PROJECT_ROOT.parent

# Add project root to Python path
sys.path.insert(0, str(PROJECT_ROOT))

from utils.utils import ensure_project_dirs, load_raw_dataset, clean_dataset, PROCESSED_DIR, REPORTS_DIR, FIGURES_DIR
from utils.features import engineer_kpis, build_post_feature_sets, aggregate_business_features
from utils.evaluation import regression_metrics, rank_models
from utils.visualization import set_plot_style, save_figure

set_plot_style()
ensure_project_dirs()

RAW_DATA_PATH = Path("../jsons/all_final_appended.json")
if not RAW_DATA_PATH.exists():
    RAW_DATA_PATH = PROJECT_ROOT / "synthetic_generator" / "synthetic_social_media_posts.csv"
KPI_PATH = PROCESSED_DIR / "kpi_dataset.csv"

## Load KPI Dataset and Weekly Aggregation

In [2]:
if KPI_PATH.exists():
    df = pd.read_csv(KPI_PATH, parse_dates=["post_date"])
else:
    df = engineer_kpis(clean_dataset(load_raw_dataset(RAW_DATA_PATH)))
df.head()

,business_name,sector,followers_count,post_date,posting_hour,day_of_week,month,post_type,caption_text,caption_length,...,is_reel,posting_time_bin,caption_length_bin,hashtags_count_bin,emoji_count_bin,engagement_level,business_size_bin,high_engagement,high_view_rate,high_comment_rate
0,LOFT Palestine,Fashion,4392.0,2026-03-31,0.0,Tuesday,3,reel,إطلالة أنثوية بلمسة عصرية ✨ من LOFT. للطلب الت...,88,...,1,night,medium,none,few,medium,small,0,0,0
1,LOFT Palestine,Fashion,4392.0,2026-03-18,0.0,Wednesday,3,reel,اختيارك المثالي لإطلالة كاجول مميزة 🤍✨ LOFT كو...,111,...,1,night,medium,none,few,high,small,1,1,0
2,LOFT Palestine,Fashion,4392.0,2026-03-16,0.0,Monday,3,reel,ستايل شبابي بلمسة عصرية له ولها ✨ اختيارات ممي...,133,...,1,night,medium,none,few,high,small,1,1,1
3,LOFT Palestine,Fashion,4392.0,2026-03-07,0.0,Saturday,3,reel,ستايل يناسبك، وتجربة تستحق الزيارة ✨ زورونا في...,82,...,1,night,medium,none,few,high,small,1,1,0
4,LOFT Palestine,Fashion,4392.0,2026-03-05,0.0,Thursday,3,reel,#LOFTSTYLE #Menstyle,21,...,1,night,short,few,none,high,small,1,1,0


## Overall Weekly Trends

In [3]:
weekly_trends = df.groupby("week", as_index=False).agg(
    total_engagement=("engagement", "sum"),
    avg_engagement_rate=("engagement_rate", "mean"),
    total_likes=("likes_count", "sum"),
    total_comments=("comments_count", "sum"),
    total_views=("views_count", "sum"),
    total_posts=("business_name", "size"),
).sort_values("week")

weekly_trends["engagement_growth"] = (
    weekly_trends["total_engagement"]
    .pct_change()
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

weekly_trends["trend_class"] = np.where(
    weekly_trends["engagement_growth"] > 0.10,
    "improving",
    np.where(
        weekly_trends["engagement_growth"] < -0.10,
        "declining",
        "stable"
    )
)

weekly_trends.head()

,week,total_engagement,avg_engagement_rate,total_likes,total_comments,total_views,total_posts,engagement_growth,trend_class
0,2020-05-18/2020-05-24,63.0,0.021724,47.0,16.0,0.0,1,0.000000,stable
1,2020-08-31/2020-09-06,0.0,0.000000,0.0,0.0,279.0,1,-1.000000,declining
2,2021-04-19/2021-04-25,112.0,0.011623,106.0,6.0,2977.0,1,0.000000,stable
3,2021-05-24/2021-05-30,89.0,0.009236,89.0,0.0,8752.0,1,-0.205357,declining
4,2021-10-25/2021-10-31,0.0,0.000000,0.0,0.0,853.0,1,-1.000000,declining


###  Business Weekly Aggregation

In [4]:
business_weekly = df.groupby(["business_name", "sector", "week"], as_index=False).agg(
    engagement_rate=("engagement_rate", "mean"),
    engagement=("engagement", "mean"),
    posts_count=("business_name", "size"),
).sort_values(["business_name", "week"])

business_weekly.head()

,business_name,sector,week,engagement_rate,engagement,posts_count
0,528 Fashion,Fashion,2026-04-27/2026-05-03,0.000111,0.500000,4
1,528 Fashion,Fashion,2026-05-04/2026-05-10,0.000519,2.333333,3
2,99streetwear,Fashion,2026-05-04/2026-05-10,0.000577,3.000000,9
3,Abu shukri restaurant-مطعم حمص ابو شكري,Cafes/Restaurants,2020-05-18/2020-05-24,0.021724,63.000000,1
4,Abu shukri restaurant-مطعم حمص ابو شكري,Cafes/Restaurants,2022-05-02/2022-05-08,0.031034,90.000000,1


## Rolling Window and Threshold Experiments

In [ ]:
windows = [2,3,4]
thresholds = [0.05,0.10,0.15]
rows, store = [], {}
# Hyperparameter Experimentation:
# i test different rolling window sizes and growth thresholds to find the most stable and interpretable
# configuration for business momentum classification
# The goal is not to choose values randomly, but to compare multiple settings and select the best one
# based on stability, inconsistency, and interpretability
for w in windows:
    for t in thresholds:
        tmp = business_weekly.copy().copy()
        tmp["rolling_engagement"] = tmp.groupby("business_name")["engagement_rate"].transform(lambda s: s.rolling(w, min_periods=1).mean())
        tmp["growth"] = tmp.groupby("business_name")["rolling_engagement"].pct_change().replace([np.inf,-np.inf], np.nan).fillna(0)
        tmp["trend_class"] = np.where(tmp["growth"] > t, "improving", np.where(tmp["growth"] < -t, "declining", "stable"))
        changes = tmp.groupby("business_name")["trend_class"].nunique().rename("n_states")
        tmp = tmp.merge(changes, on="business_name", how="left")
        tmp["final_class"] = np.where(tmp["n_states"] >= 3, "inconsistent", tmp["trend_class"])
        final = tmp.groupby("business_name", as_index=False).tail(1)
        rows.append({
            "rolling_window": w,
            "growth_threshold": t,
            "stable_ratio": (final["final_class"]=="stable").mean(),
            "inconsistent_ratio": (final["final_class"]=="inconsistent").mean(),
            "interpretability": 1 - abs(t-0.10) - abs(w-3)*0.05,
        })
        store[(w,t)] = tmp

exp = rank_models(pd.DataFrame(rows), higher_is_better_cols=["stable_ratio","interpretability"], lower_is_better_cols=["inconsistent_ratio"])
# Model/Configuration Ranking:
# This step ranks all tested hyperparameter combinations and selects the best configuration
# Higher stable_ratio and interpretability are preferred, while lower inconsistent_ratio is better
best = exp.iloc[0]
tmp = store[(int(best["rolling_window"]), float(best["growth_threshold"]))]
business_momentum = tmp.groupby(["business_name","sector"], as_index=False).tail(1)[["business_name","sector","week","rolling_engagement","growth","final_class"]].rename(columns={"final_class":"momentum_class","rolling_engagement":"latest_rolling_engagement_rate","growth":"latest_growth"})
business_weekly_trends= tmp[["business_name","sector","week","engagement_rate","rolling_engagement","growth","final_class"]].rename(columns={"final_class":"trend_class"})

## Save Outputs and Insights

In [6]:
business_momentum.to_csv(PROCESSED_DIR / "business_momentum.csv", index=False)
weekly_trends.to_csv(PROCESSED_DIR / "weekly_trends.csv", index=False)
business_weekly_trends.to_csv(PROCESSED_DIR / "business_weekly_trends.csv", index=False)

exp.to_csv(REPORTS_DIR / "momentum_experiments.csv", index=False)
display(exp)
display(business_momentum.head(15))
print("Insight: use declining/inconsistent flags for immediate coaching priorities.")

,rolling_window,growth_threshold,stable_ratio,inconsistent_ratio,interpretability,composite_score
0,4,0.10,0.430769,0.338462,0.95,2.357143
1,3,0.15,0.430769,0.338462,0.95,2.357143
2,3,0.10,0.384615,0.369231,1.00,2.071429
3,4,0.15,0.430769,0.323077,0.90,2.000000
4,2,0.15,0.400000,0.369231,0.90,1.238095
5,2,0.10,0.369231,0.400000,0.95,1.119048
6,3,0.05,0.338462,0.415385,0.95,0.642857
7,4,0.05,0.353846,0.384615,0.90,0.595238
8,2,0.05,0.338462,0.430769,0.90,0.000000


,business_name,sector,week,latest_rolling_engagement_rate,latest_growth,momentum_class
1,528 Fashion,Fashion,2026-05-04/2026-05-10,0.000315,1.833333,improving
2,99streetwear,Fashion,2026-05-04/2026-05-10,0.000577,0.000000,stable
5,Abu shukri restaurant-مطعم حمص ابو شكري,Cafes/Restaurants,2022-09-12/2022-09-18,0.024713,-0.063181,stable
6,Ahmed Sh Ajwa,Influencers,2025-12-29/2026-01-04,0.000297,0.000000,stable
8,Al Jazeera English,Influencers,2025-01-13/2025-01-19,0.008791,0.708481,improving
17,Angel shop PS,Fashion,2026-04-13/2026-04-19,0.007982,-0.093306,inconsistent
21,Baby Bump,Fashion,2026-04-06/2026-04-12,0.000440,1.200000,inconsistent
22,Bella Kids,Fashion,2026-05-04/2026-05-10,0.000597,0.000000,stable
23,Bravo Supermarket,Supermarkets,2026-04-27/2026-05-03,0.001556,0.000000,stable
30,EUROMODA,Fashion,2023-10-02/2023-10-08,0.007973,0.087558,inconsistent


Insight: use declining/inconsistent flags for immediate coaching priorities.


### Business Value

This analysis translates social media metrics into actionable business insights instead of only generating numeric outputs.

For example:
- Weekly trends show whether overall engagement is improving or declining.
- Business momentum identifies which businesses need attention.
- Momentum classification helps determine whether a business is improving, declining, stable, or inconsistent over time.

This makes the outputs directly useful for coaching decisions, dashboard alerts, and content strategy evaluation.